<img src="banner-5-coding-with-ai.png" width="100%">
<br>

# **Understanding Embeddings in Large Language Models (LLM)**

---

Embeddings are a foundational concept in natural language processing and machine learning. In the context of a language model, they convert words, sentences, or entire documents into numerical vectors of fixed size. These vectors capture the semantic meaning of the input text. In this notebook, we'll dive deep into what embeddings mean for an LLM like GPT (Generative Pre-trained Transformer).

---

**1. Introduction to Embeddings**

At its core, an embedding is a mapping from discrete objects (such as words) to vectors of real numbers.

```python
# Sample Word to Vector Representation (Hypothetical)
word = "computer"
vector_representation = [0.12, -0.58, 0.91, ...]  # a long list of numbers
```

This vector representation is useful because:
- Vectors can be input into neural networks.
- Semantically similar words will have similar vector representations.
- They allow for efficient computations to measure similarity, perform arithmetic operations, etc.

---

**2. How LLMs Use Embeddings**

LLMs typically utilize embeddings in two main phases:

1. **Embedding Layer (Input)**: Convert words/tokens into vectors.
2. **Contextual Embeddings (Hidden States)**: Capture contextual information as the model processes sequences.

The magic of LLMs like GPT is that they don't just use a static embedding for each word; the embedding changes based on the context!

---


**3. Exploring Semantic Relationships**

Embeddings can capture various semantic relationships. For example, the famous analogy "man is to king as woman is to queen" can be represented through vector arithmetic.

```python
# Hypothetical representation
vector('king') - vector('man') + vector('woman') ≈ vector('queen')
```

This showcases the depth and richness of information present in the embeddings.

---

Embeddings are a powerful way to represent text numerically, capturing rich semantic meanings in compact vectors. Through LLMs, these embeddings aren't just static but evolve based on context, providing a deep understanding of language nuances.



# Using Embeddings to answer questions about a local document

One of the ways we can use embeddings is to answer questions about a local document. For example, given a PDF file, we can extract the content, split it into smaller documents (e.g., pages), embed them, and then perform semantic search to answer questions about the document.

To process a PDF and extract information for embedding, you would typically use the `PyPDF2` library. Then, you can use FAISS and OpenAI's embeddings (or embeddings from any other model) to do a semantic search. Below is a more detailed and realistic example of how to do this.  We will use Scikit-Learn library's nearest neighbor algorithm to perform semantic search on the embeddings we get from SckiKit learn's embedding support.

---

**Embedding and Semantic Search on PDF Content using Scikit-Learn**

---

**Step 1:** Import Necessary Libraries

First, let's import the required libraries.


In [ ]:
import numpy as np
import pandas as pd


---

**Step 2:** Extract Content from PDF

We'll use `PyPDF2` to extract content from the given PDF file, "ThePragmaticProgrammer.pdf".


In [ ]:
from PyPDF2 import PdfReader

def extract_text_from_pdf(pdf_path):
    with open(pdf_path, 'rb') as file:
        reader = PdfReader(file)
        text = " ".join([page.extract_text() for page in reader.pages])
    return text

pdf_content = extract_text_from_pdf("ThePragmaticProgrammer.pdf")

In [ ]:
f"{len(pdf_content)} characters extracted from PDF."

In [ ]:
print(pdf_content[:1000])



---

**Step 3:** Split PDF Content into Documents

For simplicity, we're blindly chunking the document
into smaller documents (in this case, approximately 1500-token chunks with a 50 token overlap).
In a real-world scenario, you would want to split the document into meaningful chunks (e.g., pages, paragraphs, sections, etc.).


Here is a chunking function that we can use for our PDF.
It is capable of chunking and overlapping so that we don't miss any context between chunks.

In [ ]:
from typing import List
import tiktoken as tkn

def num_tokens_from_string(string: str, model_name: str="gpt-3.5-turbo") -> int:
    """Returns the number of tokens in a text string."""
    encoding = tkn.encoding_for_model(model_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

def chunk_prompt(prompt: str, chunk_size: int = 1500, overlap: int = 50) -> List[str]:
    """
    Splits a prompt into chunks of approximately `chunk_size` tokens, with a given overlap.

    Parameters:
    - prompt (str): The text to be chunked.
    - chunk_size (int): The desired number of tokens for each chunk.
    - overlap (int): The number of tokens for overlap between chunks.

    Returns:
    - List[str]: A list of prompt chunks.
    """

    encoding = tkn.encoding_for_model("gpt-3.5-turbo")
    tokens = list(encoding.encode(prompt))

    if len(tokens) <= chunk_size:
        return [prompt]

    chunks = []
    position = 0

    while position < len(tokens):
        start_pos = max(0, position - overlap)
        end_pos = min(position + chunk_size, len(tokens))

        chunk_tokens = tokens[start_pos:end_pos]
        chunk_text = ''.join(encoding.decode_bytes(chunk_tokens).decode('utf-8', errors='ignore'))

        chunks.append(chunk_text)

        position += chunk_size

    return chunks



Let's use that chunk_prompt function to chunk the PDF file.

In [ ]:

documents = chunk_prompt(pdf_content, chunk_size=500, overlap=50)

print(f"Original Document length: {len(pdf_content)} chars")
print(f"Number of Split Documents: {len(documents)} with overlap of 50 tokens.")
print(f"First Document length: {len(documents[0])} chars, {num_tokens_from_string(documents[0])} tokens")
total_document_length = f"{sum([len(d) for d in documents])} chars"
print(f"Total Document length: {total_document_length} chars")

for d in documents:
    print(len(d), num_tokens_from_string(d))


---

**Step 4:** Embed the Documents

We'll obtain embeddings for each document. This can be done in batches of up to 2048 documents at a time. We'll use sentence-transformers `all-MiniLM-L6-v2` model to embed the documents. This model is trained on a large corpus of text and is able to capture rich semantic information.  We will do 20 documents at a time.


In [ ]:

EMBEDDING_MODEL="all-MiniLM-L6-v2"

class LocalEmbeddingGenerator():
    """Local embedding generator using sentence-transformers."""

    def __init__(self, model_name: str = EMBEDDING_MODEL):
        self.model_name = model_name
        self._model = None
        self._dimension = None

    def _load_model(self):
        """Lazy load the sentence transformer model."""
        if self._model is None:
            try:
                from sentence_transformers import SentenceTransformer
                print(f"Loading local embedding model: {self.model_name}")
                self._model = SentenceTransformer(self.model_name, device="cpu")
                # Test embedding to get dimension
                test_embedding = self._model.encode(["test"], normalize_embeddings=True)
                self._dimension = test_embedding.shape[1]
                print(f"Local embedding model loaded. Dimension: {self._dimension}")
            except ImportError:
                raise ImportError("sentence-transformers is required for local embeddings. Install with: pip install sentence-transformers")
            except Exception as e:
                raise RuntimeError(f"Failed to load local embedding model {self.model_name}: {e}")

    def generate_embeddings(self, texts: List[str]) -> List[List[float]]:
        """Generate embeddings for a list of texts."""
        self._load_model()
        embeddings = self._model.encode(texts, batch_size=32, normalize_embeddings=True)
        return embeddings.tolist()

    def generate_single_embedding(self, text: str) -> List[float]:
        """Generate embedding for a single text."""
        self._load_model()
        embedding = self._model.encode([text], normalize_embeddings=True)
        return embedding[0].tolist()

    @property
    def embedding_dimension(self) -> int:
        """Return the dimension of the embeddings."""
        if self._dimension is None:
            self._load_model()
        return self._dimension



In [ ]:

# calculate embeddings
BATCH_SIZE = 20  # how many separate pieces of text do we embed at a time?
embeddings = []

client = LocalEmbeddingGenerator(model_name=EMBEDDING_MODEL)

for batch_start in range(0, len(documents), BATCH_SIZE):
    batch_end = batch_start + BATCH_SIZE
    batch = documents[batch_start:batch_end]
    print(f"Batch {batch_start} to {batch_end - 1}")
    response = client.generate_embeddings(batch)
    assert (len(response) == BATCH_SIZE or
            (batch_end >= len(documents) and len(response) == len(documents) - batch_start)), \
        f"Expected {BATCH_SIZE} embeddings, got {len(response)}"
    embeddings.extend(response)

df = pd.DataFrame({"text": documents, "embedding": embeddings})


In [ ]:
# We are using a data frame to visualize the data here.

print(df.head())
df.shape

Let's use SciKit-Learn's Nearest Neighbors algorithm to perform semantic search on the embeddings. We'll use the `ball_tree` algorithm, which is a fast implementation of the k-nearest neighbors algorithm.


In [ ]:
from sklearn.neighbors import NearestNeighbors

nbrs = NearestNeighbors(n_neighbors=5, algorithm='ball_tree').fit(embeddings)

Let's now try to ask some questions about the document and see if we can get the right answers.

In [ ]:
query = "How should one approach debugging?"
# Example new embedding
response = client.generate_single_embedding(text=query)
query_embedding = np.array(response).reshape(1, -1)
print(query_embedding.shape)

In [ ]:
distances, indices = nbrs.kneighbors(query_embedding)

print("Nearest Neighbors Indices:", indices)
print("Distances:", distances)

count = 0
for idx in indices[0]:
    print("""[{idx}]@{distance} {doc}""".format(idx=idx, distance=distances[0][count], doc=documents[idx].replace("\n", " ")))
    print("-" * 100)
    print("\n\n")
    count += 1


How do we know if the semantic search did a good job?
We can use a context + question prompt to OpenAI to see if the LLM can explain the question using the context. If it can, then we know that the semantic search did a good job.

In [ ]:
from openai import OpenAI
import httpx

# Configuration
proxy_url = "http://aitools.cs.vt.edu:7860/auth/register"
base_url = "http://aitools.cs.vt.edu:7860/opensource/v1"
model = "openai--gpt-oss-120b"

def aitools_proxy_login(username: str, pin: str, proxy_url: str) -> str:
    """Get token from proxy server."""
    payload = {"username": username, "pin": pin}
    with httpx.Client() as client:
        response = client.post(proxy_url, json=payload)
        response.raise_for_status()
        return response.json()["token"]

# AI Tools Proxy Login
username = "steve72"
pin = "1234"
token = aitools_proxy_login(username, pin, proxy_url)


def converse(prompt, messages=None, model="openai--gpt-oss-120b",
             max_tokens=1500,
             temperature=0,
             top_p=1,
             frequency_penalty=0,
             presence_penalty=0):

    client = OpenAI(
        base_url='http://aitools.cs.vt.edu:7860/opensource/v1',
        api_key=token
    )

    # Add the user's message to the list of messages
    if messages is None:
        messages = []

    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p,
        frequency_penalty=frequency_penalty,
        presence_penalty=presence_penalty,
    ).choices[0].message.content

    # Add the assistant's message to the list of messages
    messages.append({"role": "assistant", "content": response})

    return response, messages

prompt_template = """
Answer the following question using the context provided:
%Question:
```
{question}
```
%Context:
```
{context}
```
"""
for idx in indices[0]:
    messages = []
    prompt = prompt_template.format(question=query, context=documents[idx])
    response, messages = converse(prompt, messages)
    print(f""" [{idx}]: Explanation: {response}""")


---

**Conclusion:**


We've demonstrated how to extract content from a PDF, split it into smaller documents, embed them, and perform semantic search using Scikit-Learn and sentence-transformer's embeddings. This can be a powerful way to search through large documents or even collections of documents.

Tip: For practical deployment, always consider factors like the size of your dataset, the frequency of queries, and the desired
latency. In many real-world scenarios, batching operations, caching frequent queries, or using more specialized search libraries can significantly enhance performance and user experience. Furthermore, periodically updating embeddings can ensure that your semantic search remains relevant as the underlying content or context evolves.

---

**End of Notebook**

---
